In [1]:
%load_ext autoreload
%autoreload 2

Declaration of parameters (you must also add a tag for this cell - parameters)

In [2]:
#2. specify parameters
pipeline_params={
}
step_params={
}
substep_params={   
}

In [3]:
#3 define substep interface
from sinara.substep import NotebookSubstep, default_param_values, ENV_NAME, PIPELINE_NAME, ZONE_NAME, STEP_NAME, RUN_ID, ENTITY_NAME, ENTITY_PATH, SUBSTEP_NAME

substep = NotebookSubstep(pipeline_params, step_params, substep_params, **default_param_values("params/step_params.json"))

substep.interface(
    inputs =    
    [ 
      { STEP_NAME: "data_prep", ENTITY_NAME: "test_data"},
      { STEP_NAME: "data_prep", ENTITY_NAME: "test_config"},
      { STEP_NAME: "model_pack", ENTITY_NAME: "bento_service"}    
    ],
    
    tmp_outputs =
    [
        { ENTITY_NAME: "load_test_data" },
        { ENTITY_NAME: "test_data" },
        { ENTITY_NAME: "test_config" },
        { ENTITY_NAME: "model" }
    ]
)

substep.print_interface_info()

substep.exit_in_visualize_mode()

**INPUTS:**


[{'user.yolox_mmdet.test.data_prep.test_data': '/data/home/jovyan/yolox_mmdet/test/data_prep/run-23-10-30-153542/test_data'},
 {'user.yolox_mmdet.test.data_prep.test_config': '/data/home/jovyan/yolox_mmdet/test/data_prep/run-23-10-30-153542/test_config'},
 {'user.yolox_mmdet.test.model_pack.bento_service': '/data/home/jovyan/yolox_mmdet/test/model_pack/run-23-10-30-160331/bento_service'}]




**TMP OUTPUTS:**


[{'tmp:user.yolox_mmdet.test.model_eval.load_test_data': '/data/tmp/user/yolox_mmdet/test/model_eval/run-23-10-31-131614/load_test_data'},
 {'tmp:user.yolox_mmdet.test.model_eval.test_data': '/data/tmp/user/yolox_mmdet/test/model_eval/run-23-10-31-131614/test_data'},
 {'tmp:user.yolox_mmdet.test.model_eval.test_config': '/data/tmp/user/yolox_mmdet/test/model_eval/run-23-10-31-131614/test_config'},
 {'tmp:user.yolox_mmdet.test.model_eval.model': '/data/tmp/user/yolox_mmdet/test/model_eval/run-23-10-31-131614/model'}]




In [4]:
#4 get substep.interface
inputs_data_prep = substep.inputs(step_name = "data_prep")
inputs_model_pack = substep.inputs(step_name = "model_pack")
tmp_outputs = substep.tmp_outputs()

In [5]:
#5 run spark
from sinara.spark import SinaraSpark

spark = SinaraSpark.run_session(0)
SinaraSpark.ui_url()

Session is run


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
23/10/31 13:16:16 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


'http://localhost:4040'

#### Loading a trained model from bento_service
(weights, configs)

In [6]:
import os.path as osp
import os
from pathlib import Path
from sinara.bentoml import load_bentoservice

# read trained model
print('read trained model')    
bento_service = load_bentoservice(inputs_model_pack.bento_service)

read trained model
[2023-10-31 13:16:23,298] INFO - Detected zipimporter <zipimporter object "/home/jovyan/work/yolox_mmdet/4_model_eval/tmp/run-23-10-31-131614/bento_service/Model_YOLOX_Pack/zipimports/py4j-src.zip/">
[2023-10-31 13:16:23,374] WARNING - Python 3.9.10 found in current environment is not officially supported by BentoML. The docker base image used is'bentoml/model-server:0.13.2' which will use conda to install Python 3.9.10 in the build process. Supported Python versions are: f3.6, 3.7, 3.8


##### Save binary model weights and config to tmp_outputs.model

In [7]:
model_name = bento_service.artifacts["model_name"].get()
model_path = osp.join(tmp_outputs.model, f"latest.pth")
cfg_path = osp.join(tmp_outputs.model, f"last_cfg.py")
bento_service.artifacts["model"].save(model_path)
bento_service.artifacts["config"].save(cfg_path)

#### Loading test datasets (from step data_prep)

In [8]:
from sinara.store import SinaraStore

# copy config from previos step to outputs
SinaraStore.copy_store_files_to_tmp(store_dir=inputs_data_prep.test_data, tmp_dir=tmp_outputs.load_test_data)
SinaraStore.copy_store_files_to_tmp(store_dir=inputs_data_prep.test_config, tmp_dir=tmp_outputs.test_config)

In [9]:
from functools import partial

### Save dataset from parquet to files
def save_file(row, tmp_dir): 
    total_img_path = osp.join(tmp_dir, row.file_names)
    os.makedirs(osp.dirname(total_img_path), exist_ok=True)
    with open(total_img_path, 'wb') as f_id:
        f_id.write(row.files_binary)

In [10]:
%%time

# LOAD TEST Images

print(f"spark read start")
df_spark = spark.read.parquet(tmp_outputs.load_test_data)
df_spark.foreach(partial(save_file, tmp_dir=tmp_outputs.test_data))

spark read start


CPU times: user 6.29 ms, sys: 1.95 ms, total: 8.24 ms
Wall time: 4.53 s


In [11]:
#9 stop spark
SinaraSpark.stop_session()